# Hướng dẫn chi tiết: Xây dựng CRNN từ con số 0
Notebook này được thiết kế để giải thích **chi tiết từng bước** cách xây dựng pipeline cho mô hình nhận dạng chữ (OCR) với kiến trúc **CRNN + CTC Loss** mà không dùng các module đóng gói sẵn. Chúng ta sẽ làm mọi thứ thủ công để bạn dễ hiểu nhất.

---

## Bước 1: Import Thư viện
Đầu tiên ta import PyTorch và các thư viện hỗ trợ xử lý ảnh (OpenCV, Matplotlib).

In [ ]:
import os
import glob
import json
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights

## Bước 2: Chuẩn bị Bảng ký tự (Charset) và Encoding
Trong OCR, mô hình dự đoán ra chuỗi các con số (Index). Ta cần có một từ điển để ánh xạ (map) từ ký tự (chữ 'a', 'b', 'c') sang ID (1, 2, 3), và ngược lại.

**Đặc biệt với CTC Loss:** CTC Loss yêu cầu 1 ID đặc biệt gọi là **Blank Token** (Ký tự trống). Ký tự này thường đặt là Index 0. Mục đích là để phân biệt giữa các chữ cái bị lặp liền kề nhau (ví dụ: chữ 'o' và 'o' trong từ 'book').

In [ ]:
# Đọc bảng chữ cái từ file của dự án
charset_path = "../data/charset.txt"
with open(charset_path, 'r', encoding='utf-8') as f:
    chars = f.read()

# Tạo Dictionary mapping (char -> id) và (id -> char)
char2idx = {}
idx2char = {}

for i, char in enumerate(chars):
    idx = i + 1  # Bắt đầu từ 1. Vì 0 sẽ được dùng cho CTC Blank Token.
    char2idx[char] = idx
    idx2char[idx] = char

num_classes = len(chars)  # Số ký tự (không tính blank)
print(f"Tổng số ký tự trong từ điển: {num_classes}")
print(f"ID của chữ 'a': {char2idx.get('a', 'Không có')}")

### Hàm Mã hoá (Encode)
Ta sẽ chuyển chuỗi text thành list các con số dựa vào từ điển vừa tạo.

In [ ]:
def encode_text(text):
    encoded = []
    for char in text:
        if char in char2idx:
            encoded.append(char2idx[char])
    return encoded

# Test thử
sample_text = "Hello"
print(f"Chuỗi '{sample_text}' mã hoá thành: {encode_text(sample_text)}")

## Bước 3: Dataset - Viết class load dữ liệu thủ công
PyTorch yêu cầu phải kế thừa class `Dataset` và thiết lập 2 hàm chính: `__len__` (tổng số mẫu) và `__getitem__` (lấy 1 mẫu).

Ở đây, ta đọc danh sách file JSON chứa nhãn trong `data/synthetic/train`. File JSON sẽ chỉ đường dẫn tới ảnh `.png` tương ứng.

In [ ]:
class ManualSTRDataset(Dataset):
    def __init__(self, data_dir, is_train=True):
        self.samples = []
        labels_file = os.path.join(data_dir, 'labels.jsonl')
        
        if os.path.exists(labels_file):
            with open(labels_file, 'r', encoding='utf-8') as f:
                for line in f:
                    item = json.loads(line)
                    img_path = os.path.join(data_dir, item['file'])
                    if os.path.exists(img_path):
                        self.samples.append({'img_path': img_path, 'label': item['label']})
        
        print(f"Đã load {len(self.samples)} mẫu từ {data_dir}")
        
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img_path = sample['img_path']
        text = sample['label']
        
        # 1. Đọc ảnh bằng OpenCV
        img = cv2.imread(img_path)
        
        # 2. Xử lý ảnh: Đưa về thang độ xám (Grayscale) và Resize về chiều cao = 32.
        # Tại sao cao 32? Vì các mạng CNN (như VGG, ResNet) thường chia kích thước ảnh 2^5 = 32 lần.
        # Nếu H=32, thì đi qua CNN chiều cao sẽ triệt tiêu về 1.
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Tính toán chiều rộng mới tỷ lệ thuận với chiều cao 32
        h, w = img.shape
        new_w = int(w * (32 / h))
        img = cv2.resize(img, (new_w, 32))
        
        # 3. Chuẩn hoá điểm ảnh về khoảng [-1, 1]
        img = img.astype(np.float32) / 255.0
        img = (img - 0.5) / 0.5
        
        # 4. Chuyển thành PyTorch Tensor, thêm chiều Channel (1, H, W)
        img_tensor = torch.from_numpy(img).unsqueeze(0)
        
        # 5. Mã hoá Label
        label_encoded = encode_text(text)
        
        return img_tensor, label_encoded, text

# Test thử Dataset
train_dataset = ManualSTRDataset("../data/synthetic/train")
img, label, text = train_dataset[0]
print(f"\nSample 0 - Text: {text}")
print(f"Encoded Label: {label}")
print(f"Image Shape: {img.shape}")

### Hàm Gộp Batch (Collate Function)
Trong DataLoader, ta gộp nhiều ảnh thành 1 Batch để đưa vào mạng CNN. Tuy nhiên, **chiều rộng (W) của các ảnh thường không bằng nhau**. Và **độ dài của các nhãn (Labels) cũng không bằng nhau**.

Hàm `collate_fn` giúp ta Padding (điền số 0) để các ảnh và các label trong cùng 1 batch có kích thước bằng với cái dài nhất.

In [ ]:
def custom_collate(batch):
    # Batch là 1 list các tuple (img_tensor, label_encoded, text) do __getitem__ trả về
    
    # Tìm chiều rộng ảnh lớn nhất và chiều dài label lớn nhất trong batch này
    max_w = max([item[0].shape[2] for item in batch])
    max_label_len = max([len(item[1]) for item in batch])
    
    images_padded = []
    labels_padded = []
    label_lengths = []
    
    for img, lbl, txt in batch:
        # Pad ảnh (bằng số 0)
        pad_w = max_w - img.shape[2]
        img_pad = torch.nn.functional.pad(img, (0, pad_w, 0, 0), value=0)
        images_padded.append(img_pad)
        
        # Pad label (bằng số 0 - mặc định 0 là Blank)
        pad_lbl = lbl + [0] * (max_label_len - len(lbl))
        labels_padded.append(pad_lbl)
        label_lengths.append(len(lbl))
        
    # Gộp thành các tensor lớn chung của Batch
    images_tensor = torch.stack(images_padded)               # (Batch_Size, 1, 32, max_W)
    labels_tensor = torch.tensor(labels_padded, dtype=torch.long) # (Batch_Size, max_label_len)
    label_lengths = torch.tensor(label_lengths, dtype=torch.long)
    
    return images_tensor, labels_tensor, label_lengths

# Tạo DataLoader
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=custom_collate)

# Rút thử 1 Batch
images_b, labels_b, len_b = next(iter(train_loader))
print(f"Batch Images Shape: {images_b.shape}")
print(f"Batch Labels Shape: {labels_b.shape}")
print(f"Lengths của 4 label: {len_b}")

## Bước 4: Khởi tạo Kiến trúc Mạng CRNN
Mô hình CRNN (Convolutional Recurrent Neural Network) gồm 3 phần chính:
1. **CNN (Convolutional Layers)**: Trích xuất đặc trưng hình ảnh. Đầu vào (B, 1, 32, W) sẽ bị ép giảm chiều cao dần dần về (B, 512, 1, W'). Chúng ta dùng ResNet18 làm backbone vì nó rất sâu và mạnh.
2. **RNN (Recurrent Layers)**: Phân tích ngữ cảnh dạng chuỗi từ trái qua phải. Dùng Bi-LSTM 2 chiều.
3. **FC (Linear)**: Biến đổi Vector ở cuối về số chiều `Num_classes + 1` (đã tính Blank).

**Lưu ý kỹ thuật quan trọng**: Trong ResNet18 gốc, ảnh bị bóp nhỏ cả chiều rộng và chiều cao (stride = 2). Trong bài toán nhận dạng chữ dài, ta không muốn chiều rộng bị bóp quá ngắn. Nên ta phải can thiệp sửa Stride của ResNet thành `(2, 1)`.

In [ ]:
class CustomCRNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        
        # 1. Khởi tạo CNN (ResNet18)
        backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        
        # Mặc định layer2, layer3, layer4 của resnet dùng stride (2, 2) tức là giảm một nửa cả Ngang lẫn Dọc.
        # Thay vì (2, 2), ta chỉnh thành (2, 1) để CHỈ thu hẹp chiều cao, giữ nguyên chiều ngang.
        backbone.layer2[0].conv1.stride = (2, 1)
        backbone.layer2[0].downsample[0].stride = (2, 1)
        
        backbone.layer3[0].conv1.stride = (2, 1)
        backbone.layer3[0].downsample[0].stride = (2, 1)
        
        backbone.layer4[0].conv1.stride = (2, 1)
        backbone.layer4[0].downsample[0].stride = (2, 1)
        
        # Gom các lớp CNN thành cục
        self.cnn = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool,
            backbone.layer1, backbone.layer2, backbone.layer3, backbone.layer4
        )
        
        # 2. Khởi tạo RNN (BiLSTM)
        self.rnn = nn.LSTM(input_size=512, hidden_size=256, num_layers=2, bidirectional=True, batch_first=True)
        
        # 3. Head (Classifier)
        # BiLSTM 2 chiều nên ouput rnn = 256 * 2 = 512.
        self.fc = nn.Linear(512, num_classes + 1) # +1 cho Blank Token

    def forward(self, x):
        # x là ảnh (B, 1, 32, W). Cần lặp kênh thành 3 màu vì ResNet cần RGB
        x = x.repeat(1, 3, 1, 1)
        
        # Đi qua CNN
        feat = self.cnn(x)              # Hình dáng: (Batch, 512, Chiều_cao=1, Chiều_rộng_mới=W')
        
        # Ép bẹp chiều cao vì nó = 1
        feat = feat.squeeze(2)          # Hình dáng: (Batch, 512, W')
        
        # RNN muốn nhận đầu vào chuẩn (Batch, Thời_Gian, Feature) tức là (Batch, W', 512)
        # Ta dùng permute để xoay
        feat = feat.permute(0, 2, 1)    
        
        # Đi qua RNN
        out, _ = self.rnn(feat)         # Hình dáng: (Batch, W', 512)
        
        # Phân loại để xuất ra xác suất chữ cái (Logits)
        logits = self.fc(out)           # Hình dáng: (Batch, W', 236)
        return logits

# Khởi tạo model thử
model = CustomCRNN(num_classes=num_classes)
logits_b = model(images_b)
print(f"Hình dáng đầu ra (Logits): {logits_b.shape}")

## Bước 5: Hàm Decode Model Output (Tham Lam - Greedy)
Model xuất ra 1 bảng Logits chứa xác suất tại mỗi Timestep (Mỗi pixel rộng).
Ta sẽ tìm vị trí có xác suất cao nhất ở từng bước (Argmax).

CTC yêu cầu thuật toán xoá bỏ (collapse): Bỏ đi các Blank Token (Index 0) và gộp 2 chữ lặp liên tiếp (ví dụ: `a-aa-` => `a a`).

In [ ]:
def ctc_greedy_decode(logits, idx2char):
    # Logits: (Batch, Timestep, Class)
    _, preds = logits.max(2) # Lấy index lớn nhất (Batch, Timestep)
    preds = preds.cpu().numpy()
    
    result_strings = []
    for b in range(preds.shape[0]):
        pred_seq = preds[b]
        chars = []
        
        # Giải thuật CTC Collapse
        for i, char_idx in enumerate(pred_seq):
            # Bỏ số 0 (blank)
            if char_idx != 0:
                # Nếu nó giống ký tự đứng NGAY TRƯỚC nó, thì bỏ qua
                if i > 0 and char_idx == pred_seq[i - 1]:
                    continue
                # Dịch ngược index ra chữ cái
                chars.append(idx2char[char_idx])
                
        result_strings.append("".join(chars))
    return result_strings

# Thử decode một cái Model rỗng chưa train:
print("Dự đoán trước khi Train:", ctc_greedy_decode(logits_b, idx2char))

## Bước 6: Vòng lặp Huấn luyện chi tiết (Train Loop)
Bây giờ ta sẽ gắn mọi thứ lại với nhau để Train mô hình.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# Hàm Loss CTCLoss
# Zero_infinity=True giúp bỏ qua các batch bị lỗi do Label dài hơn Timestep
criterion = nn.CTCLoss(blank=0, zero_infinity=True, reduction="mean")
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 2
print("===== BẮT ĐẦU TRAINING =====")

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    
    for batch_idx, (images, labels, labels_len) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        labels_len = labels_len.to(device)
        
        optimizer.zero_grad()
        
        # 1. Forward
        outputs = model(images)  # (Batch, Timestep, Class)
        
        # 2. Định dạng lại cho CTCLoss (Nó ép buộc phải là [Timestep, Batch, Class])
        log_probs = outputs.log_softmax(2).permute(1, 0, 2)
        
        # Tính Timestep cho từng ảnh trong batch (trong bài này đều bằng nhau = output.size(1))
        input_lengths = torch.full((images.size(0),), outputs.size(1), dtype=torch.long)
        
        # 3. Tính Loss
        loss = criterion(log_probs, labels.cpu(), input_lengths, labels_len.cpu())
        
        # 4. Cập nhật Model
        loss.backward()
        # Chống bùng nổ Gradient (Gradient Clipping)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        
        if batch_idx % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] - Batch [{batch_idx}] - Loss: {loss.item():.4f}")
            
    print(f"=> Loss trung bình Epoch {epoch+1}: {epoch_loss/len(train_loader):.4f}\n")

## Bước 7: Trực quan lại kết quả
Sau khi Train, model sẽ khôn hơn một chút. Cùng xem nó dự đoán như thế nào.

In [ ]:
model.eval()
with torch.no_grad():
    outputs = model(images_b.to(device))
    preds = ctc_greedy_decode(outputs, idx2char)

fig, axes = plt.subplots(4, 1, figsize=(10, 8))
for i in range(4):
    img = images_b[i].cpu().numpy()
    img = np.transpose(img, (1, 2, 0))
    img = img * 0.5 + 0.5  # Bỏ chuẩn hoá
    img = np.clip(img, 0, 1)
    
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f"Dự đoán: {preds[i]}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()